# Chấm test cuối cùng

Đây là notebook duy nhất đọc nhãn `datasets/splits/test.csv`. Mọi cấu hình được đọc từ `experiments/final_candidates.json`, không có bước tune nào ở đây. Họ so sánh được định nghĩa trong `experiment_plan.md`.

Điểm test của 4 mô hình đã có sẵn từ trước trong `comparison_val_test.csv`, nên đây không phải lần đầu test được nhìn.

Chốt chặn: notebook dừng nếu kết quả đã tồn tại, trừ khi đặt `FINAL_TEST_REPRO=1` để tái lập đúng cấu hình đã khóa.

In [1]:
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_v, "1")
import hashlib, json, math, platform, re, subprocess, time
from collections import Counter, namedtuple
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import scipy.sparse as sps
import sklearn
import torch
from scipy.stats import binomtest
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

SEED = 42
DEVICE = torch.device("cuda")
DTYPE = torch.float64
ROOT = Path.cwd()
while not (ROOT / "Data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
EXP = ROOT / "models" / "Naive_Bayes" / "experiments"
SPLITS = ROOT / "models" / "Naive_Bayes" / "datasets" / "splits"
LABELS = ["Negative", "Neutral", "Positive"]
L2I = {l: i for i, l in enumerate(LABELS)}
C = len(LABELS)

LOCKED = {
    "models/Naive_Bayes/datasets/splits/train.csv": "c04400673765183a0c904d447b611f48b74389c1b68059d517237d4cdab38505",
    "models/Naive_Bayes/datasets/splits/val.csv": "155c13a615172a7b06c4f2b647e222736d0354dcb603ad90c0c027ecd67f31c7",
    "models/Naive_Bayes/datasets/splits/test.csv": "6135fb08b390b6743968a1fb1dde8319977e3297bad656f00e1d669f972b9ae2",
    "Data/final_labeled_clean_v3.xlsx": "ea6c578763a8396f4c065696e50df0431a0a978cfff40399818e2a7b4ef7a89c",
}
for rel, want in LOCKED.items():
    assert hashlib.sha256((ROOT / rel).read_bytes()).hexdigest() == want, rel

RES_FILE = EXP / "final_test_results.csv"
assert os.environ.get("FINAL_TEST_REPRO") == "1" or not RES_FILE.exists(), "Test đã được chấm. Đặt FINAL_TEST_REPRO=1 chỉ để tái lập."

cand_bytes = (EXP / "final_candidates.json").read_bytes()
CAND = json.loads(cand_bytes)
CAND_SHA = hashlib.sha256(cand_bytes).hexdigest()
INCLUDE_X = bool(CAND["Exploratory_X_old_bal"]["eligible_for_test"])
print("Ứng viên khóa lúc", CAND["locked_at"], "| sha256", CAND_SHA[:16], "| X_old_bal vào test:", INCLUDE_X)

Ứng viên khóa lúc 2026-11-05 21:52:04 EST | sha256 96e9fe6eccf85f10 | X_old_bal vào test: True


In [2]:
def read_split(name):
    return pd.read_csv(SPLITS / f"{name}.csv", encoding="utf-8-sig", keep_default_na=False).assign(split=name)

df = pd.concat([read_split(n) for n in ("train", "val", "test")], ignore_index=True)
xl = pd.read_excel(ROOT / "Data" / "final_labeled_clean_v3.xlsx", keep_default_na=False, usecols=["Comment ID", "Clean Text", "Clean Text Old"])
xl = xl.set_index("Comment ID").loc[df["Comment ID"]]
assert (xl["Clean Text"].astype(str).values == df["Clean Text"].astype(str).values).all()
df["Clean Text Old"] = xl["Clean Text Old"].astype(str).values
N = len(df)
y_np = df["Final Label"].map(L2I).to_numpy()
y = torch.tensor(y_np, device=DEVICE)
tr_only = torch.tensor((df.split == "train").values, device=DEVICE)
dev_m = torch.tensor((df.split != "test").values, device=DEVICE)
te_m = torch.tensor((df.split == "test").values, device=DEVICE)
te_np = te_m.cpu().numpy()
y_te = y[te_m]
gid_te = torch.tensor(pd.factorize(df.loc[te_np, "Clean Text"])[0], device=DEVICE)
print("train", int(tr_only.sum()), "| dev", int(dev_m.sum()), "| test", int(te_m.sum()), "| group test", int(gid_te.max()) + 1)

TOKEN_RE = re.compile(r"(?u)\b\w\w+\b")

def word_terms(text):
    t = TOKEN_RE.findall(text.lower())
    return [(w, 1) for w in t] + [(f"{t[i]} {t[i + 1]}", 2) for i in range(len(t) - 1)]

class Corpus:
    def __init__(self, texts):
        idx, order, d_, t_, f_ = {}, [], [], [], []
        for i, txt in enumerate(texts):
            for (w, o), c in Counter(word_terms(txt)).items():
                j = idx.get(w)
                if j is None:
                    j = idx[w] = len(idx); order.append(o)
                d_.append(i); t_.append(j); f_.append(c)
        self.doc = torch.tensor(d_, device=DEVICE); self.term = torch.tensor(t_, device=DEVICE)
        self.tf = torch.tensor(f_, dtype=DTYPE, device=DEVICE); self.order = torch.tensor(order, device=DEVICE)
        self.n_docs, self.n_terms = len(texts), len(idx)

CORP = {"clean": Corpus(df["Clean Text"].tolist())}
if INCLUDE_X:
    CORP["old"] = Corpus(df["Clean Text Old"].tolist())
Feat = namedtuple("Feat", "d t v vm")

def vectorize(cp, tr, min_df, max_order):
    ok = cp.order[cp.term] <= max_order
    df_ = torch.bincount(cp.term[tr[cp.doc] & ok], minlength=cp.n_terms).to(DTYPE)
    vm = df_ >= min_df
    m = vm[cp.term]
    d, t, f = cp.doc[m], cp.term[m], cp.tf[m]
    n = tr.sum().to(DTYPE)
    idf = torch.log((1 + n) / (1 + df_)) + 1.0
    v = (1.0 + torch.log(f)) * idf[t]
    norm2 = torch.zeros(cp.n_docs, dtype=DTYPE, device=DEVICE).index_add_(0, d, v * v)
    return Feat(d, t, v / norm2.sqrt()[d].clamp_min(1e-12), vm)

def nb_probs(cfg, tr, weights=None):
    cp = CORP[cfg["corpus"]]
    assert cfg["weighting"] == "tfidf" and not cfg["fit_prior"]
    ft = vectorize(cp, tr, cfg["min_df"], cfg["max_order"])
    k = tr[ft.d]
    vals = ft.v[k]
    w = None
    if weights == "balanced":
        cc = torch.bincount(y[tr], minlength=C).to(DTYPE)
        w = (tr.sum().to(DTYPE) / (C * cc))[y]
        vals = vals * w[ft.d[k]]
    fc = torch.zeros(C, cp.n_terms, dtype=DTYPE, device=DEVICE)
    fc.index_put_((y[ft.d[k]], ft.t[k]), vals, accumulate=True)
    s = (fc + cfg["alpha"]) * ft.vm
    lt = torch.log(s.clamp_min(1e-300)) - torch.log(s.sum(1, keepdim=True))
    e = te_m[ft.d]
    pos = torch.full((N,), -1, device=DEVICE); pos[te_m] = torch.arange(int(te_m.sum()), device=DEVICE)
    jl = torch.zeros(int(te_m.sum()), C, dtype=DTYPE, device=DEVICE)
    jl.index_add_(0, pos[ft.d[e]], ft.v[e].view(-1, 1) * lt[:, ft.t[e]].T)
    return jl

def tfidf_csr(cfg, tr):
    cp = CORP["clean"]
    ft = vectorize(cp, tr, cfg["min_df"], cfg["max_order"])
    cols = torch.full((cp.n_terms,), -1, device=DEVICE); cols[ft.vm] = torch.arange(int(ft.vm.sum()), device=DEVICE)
    return sps.csr_matrix((ft.v.cpu().numpy(), (ft.d.cpu().numpy(), cols[ft.t].cpu().numpy())), shape=(N, int(ft.vm.sum())))

train 1491 | dev 1800 | test 287 | group test 271


In [3]:
MODELS, PROBS = {}, {}

jl = nb_probs(CAND["R0"], tr_only)
MODELS["R0_train"] = jl.argmax(1)
f_hist = f1_score(y_te.cpu(), MODELS["R0_train"].cpu(), average="macro")
print("R0 fit trên train (đối chiếu lịch sử):", f_hist)
assert abs(f_hist - 0.5440343708530961) < 1e-9, "không tái lập được macro-F1 0.5440 đã công bố"

MODELS["R0_refit"] = nb_probs(CAND["R0"], dev_m).argmax(1)
jl = nb_probs(CAND["NB_best"], dev_m)
MODELS["NB_best"] = jl.argmax(1)
PROBS["NB_best"] = torch.softmax(jl, 1)

lr = CAND["LogReg"]
X = tfidf_csr(lr, dev_m)
d = dev_m.cpu().numpy()
m = LogisticRegression(C=lr["C"], class_weight=lr["class_weight"], max_iter=2000).fit(X[d], y_np[d])
PROBS["LogReg"] = torch.tensor(m.predict_proba(X[te_np]), dtype=DTYPE, device=DEVICE)
MODELS["LogReg"] = PROBS["LogReg"].argmax(1)

h = CAND["Hybrid"]
assert h["method"] == "soft_vote", h
MODELS["Hybrid"] = (h["w_nb"] * PROBS["NB_best"] + (1 - h["w_nb"]) * PROBS["LogReg"]).argmax(1)

if INCLUDE_X:
    MODELS["X_old_bal"] = nb_probs(CAND["Exploratory_X_old_bal"], dev_m, CAND["Exploratory_X_old_bal"]["weights"]).argmax(1)

def macro_f1_cm(cm):
    tp = cm.diagonal(dim1=-2, dim2=-1)
    p = tp / cm.sum(-2).clamp_min(1e-12); r = tp / cm.sum(-1).clamp_min(1e-12)
    return p, r, 2 * p * r / (p + r).clamp_min(1e-12)

B = 10000
G = int(gid_te.max()) + 1
g = torch.Generator(device=DEVICE).manual_seed(SEED)
draw = torch.randint(0, G, (B, G), generator=g, device=DEVICE)
W = torch.zeros(B, G, dtype=DTYPE, device=DEVICE).scatter_add_(1, draw, torch.ones_like(draw, dtype=DTYPE))[:, gid_te]

def boot(pred):
    oh = torch.nn.functional.one_hot(y_te * C + pred, C * C).to(DTYPE)
    return macro_f1_cm((W @ oh).view(-1, C, C))[2].mean(1)

BOOTS = {k: boot(v) for k, v in MODELS.items()}
rows, per = [], []
for name, p in MODELS.items():
    cm = torch.bincount(y_te * C + p, minlength=C * C).view(C, C).to(DTYPE)
    pr, rc, f = macro_f1_cm(cm)
    lo, hi = np.percentile(BOOTS[name].cpu().numpy(), [2.5, 97.5])
    mf = float(f.mean())
    assert abs(mf - f1_score(y_te.cpu(), p.cpu(), average="macro")) < 1e-9
    rows.append(dict(model=name, macro_f1=mf, ci95_lo=lo, ci95_hi=hi, accuracy=accuracy_score(y_te.cpu(), p.cpu()),
                     f1_negative=float(f[0]), f1_neutral=float(f[1]), f1_positive=float(f[2]), n_test=int(te_m.sum()), n_groups=G, bootstrap_B=B))
    for c in range(C):
        per.append(dict(model=name, label=LABELS[c], precision=float(pr[c]), recall=float(rc[c]), f1=float(f[c]), support=int(cm[c].sum())))
    pd.DataFrame(cm.cpu().numpy().astype(int), index=LABELS, columns=LABELS).to_csv(EXP / f"final_test_confusion_{name}.csv")
results = pd.DataFrame(rows)
results.to_csv(RES_FILE, index=False)
pd.DataFrame(per).to_csv(EXP / "final_test_per_class.csv", index=False)
print(results.round(4).to_string(index=False))

R0 fit trên train (đối chiếu lịch sử): 0.5440343708530961


    model  macro_f1  ci95_lo  ci95_hi  accuracy  f1_negative  f1_neutral  f1_positive  n_test  n_groups  bootstrap_B
 R0_train    0.5440   0.4723   0.6127    0.6760       0.4286      0.4096       0.7939     287       271        10000
 R0_refit    0.5618   0.4907   0.6295    0.6934       0.5049      0.3704       0.8103     287       271        10000
  NB_best    0.5395   0.4699   0.6051    0.6585       0.4259      0.4000       0.7925     287       271        10000
   LogReg    0.5966   0.5262   0.6618    0.7003       0.4906      0.4842       0.8150     287       271        10000
   Hybrid    0.5902   0.5194   0.6561    0.6899       0.4952      0.4742       0.8011     287       271        10000
X_old_bal    0.5789   0.5066   0.6458    0.7038       0.4808      0.4416       0.8142     287       271        10000


In [4]:
FAMILY = [("NB_best", "R0_refit"), ("Hybrid", "NB_best"), ("NB_best", "LogReg")]
if INCLUDE_X:
    FAMILY.append(("X_old_bal", "NB_best"))
fam = FAMILY if CAND["NB_best"]["alpha"] != CAND["R0"]["alpha"] or CAND["NB_best"]["min_df"] != CAND["R0"]["min_df"] else FAMILY[1:]
cmp_rows = []
for a, b in fam:
    fa = results.set_index("model").loc[a, "macro_f1"]; fb = results.set_index("model").loc[b, "macro_f1"]
    d = (BOOTS[a] - BOOTS[b]).cpu().numpy()
    lo, hi = np.percentile(d, [2.5, 97.5])
    p = max(min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean())), 1 / B)
    ca, cb = (MODELS[a] == y_te).cpu().numpy(), (MODELS[b] == y_te).cpu().numpy()
    n01, n10 = int((ca & ~cb).sum()), int((~ca & cb).sum())
    mc = binomtest(min(n01, n10), n01 + n10, 0.5).pvalue if n01 + n10 else 1.0
    cmp_rows.append(dict(a=a, b=b, f1_a=fa, f1_b=fb, delta=fa - fb, ci95_lo=lo, ci95_hi=hi, p_boot=p, a_right_b_wrong=n01, a_wrong_b_right=n10, p_mcnemar_exact=mc))
cmp_ = pd.DataFrame(cmp_rows)
o = cmp_["p_boot"].argsort().values
adj, cur = np.zeros(len(cmp_)), 0.0
for i, j in enumerate(o):
    cur = max(cur, min(1.0, (len(cmp_) - i) * cmp_["p_boot"].values[j]))
    adj[j] = cur
cmp_["p_holm"] = adj
cmp_["significant_holm_0.05"] = cmp_["p_holm"] < 0.05
cmp_.to_csv(EXP / "final_test_comparisons.csv", index=False)
print(cmp_.round(4).to_string(index=False))

out = pd.DataFrame({"Comment ID": df.loc[te_np, "Comment ID"].values, "y_true": df.loc[te_np, "Final Label"].values})
for k, v in MODELS.items():
    out[f"pred_{k}"] = [LABELS[i] for i in v.tolist()]
out.to_csv(EXP / "final_test_predictions.csv", index=False)
info = {"candidates_sha256": CAND_SHA, "seed": SEED, "bootstrap_B": B, "n_test": int(te_m.sum()), "n_groups_test": G,
        "git_commit": subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip(),
        "python": platform.python_version(), "torch": torch.__version__, "numpy": np.__version__, "scipy": scipy.__version__,
        "sklearn": sklearn.__version__, "gpu": torch.cuda.get_device_name(0), "x_old_bal_included": INCLUDE_X,
        "scored_at": time.strftime("%Y-%m-%d %H:%M:%S %Z"), "repro_run": os.environ.get("FINAL_TEST_REPRO") == "1"}
(EXP / "final_test_run_info.json").write_text(json.dumps(info, ensure_ascii=False, indent=2), encoding="utf-8")

        a        b   f1_a   f1_b   delta  ci95_lo  ci95_hi  p_boot  a_right_b_wrong  a_wrong_b_right  p_mcnemar_exact  p_holm  significant_holm_0.05
  NB_best R0_refit 0.5395 0.5618 -0.0224  -0.0699   0.0264  0.3750               12               22           0.1214  0.5400                  False
   Hybrid  NB_best 0.5902 0.5395  0.0507   0.0144   0.0904  0.0066               14                5           0.0636  0.0264                   True
  NB_best   LogReg 0.5395 0.5966 -0.0571  -0.1217   0.0088  0.0914               21               33           0.1337  0.2742                  False
X_old_bal  NB_best 0.5789 0.5395  0.0394  -0.0307   0.1081  0.2700               34               21           0.1048  0.5400                  False


471